# A larger PBMC dataset

Try a separate 10x dataset with 11,769 cells and 33,538 genes. This example reads the full sparse count matrix, then uses Scanpy to prepare an embedding.

Unreleased preview for Scanpy 1.11.5. Install from the project folder with `pip install -e '.[scanpy]'` and use that environment as your Jupyter kernel. Use Python 3.11 or newer. The first run downloads a 94 MB archive and computes the analysis below.


In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen
from shutil import copyfileobj
from hashlib import file_digest
import tarfile

import scanpy as sc
import genome_spy.integrations.scanpy as gsc

In [ ]:
folder = Path("data/pbmc10k")
folder.mkdir(parents=True, exist_ok=True)
archive = folder / "pbmc_10k_v3_filtered_feature_bc_matrix.tar.gz"
url = "https://cf.10xgenomics.com/samples/cell-exp/3.0.0/pbmc_10k_v3/pbmc_10k_v3_filtered_feature_bc_matrix.tar.gz"
if not archive.exists():
    request = Request(url, headers={"User-Agent": "genome-spy-python-docs"})
    with urlopen(request, timeout=60) as response, archive.open("wb") as output:
        copyfileobj(response, output)
with archive.open("rb") as source:
    checksum = file_digest(source, "sha256").hexdigest()
if checksum != "060af9e4370c6ad877e2987cf8cdc63a6d5c84e57875580f2be6c8e90748cf63":
    raise ValueError("The downloaded PBMC10k archive has an unexpected checksum.")
with tarfile.open(archive) as source:
    source.extractall(folder, filter="data")
raw = sc.read_10x_mtx(folder / "filtered_feature_bc_matrix")

## Inspect the original counts

Compare the most abundant genes before filtering cells. Hover for summaries and outlying cells.


In [ ]:
large_highest = gsc.pl.highest_expr_genes(
    raw,
    n_top=20,
    width=400,
    height=380,
    zoom=True,
    show=False,
)
large_highest

## Prepare the embedding with Scanpy

Filter cells with fewer than 200 detected genes, then normalize counts and select variable genes. Scanpy computes PCA, neighbors and UMAP once; the plotting calls below read those results.


In [ ]:
adata = raw.copy()
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
)
sc.pp.normalize_total(adata, target_sum=10_000)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.pp.pca(adata, mask_var="highly_variable", random_state=0)
sc.pp.neighbors(adata, n_pcs=30, random_state=0)
sc.tl.umap(adata, random_state=0)

## Inspect cell quality

Compare total counts with detected genes. Color shows the mitochondrial percentage; hover and zoom to inspect cells.


In [ ]:
large_scatter = gsc.pl.scatter(
    adata,
    x="total_counts",
    y="n_genes_by_counts",
    color="pct_counts_mt",
    size=4,
    width=360,
    height=280,
    zoom=True,
    show=False,
)
large_scatter

## Plot marker expression

Show normalized expression for two markers using the stored UMAP coordinates. Small points grow as you zoom.


In [ ]:
large_umap = gsc.pl.umap(
    adata,
    color=["CD3D", "MS4A1"],
    size=4,
    width=240,
    height=260,
    zoom=True,
    show=False,
)
large_umap

## Inspect PCA variance

Inspect the PCA results used for the neighbor graph. This call does not rerun PCA.


In [ ]:
large_variance = gsc.pl.pca_variance_ratio(
    adata,
    n_pcs=20,
    log=False,
    width=360,
    height=240,
    show=False,
)
large_variance

:::{admonition} Data use and provenance
:class: note

[10k PBMCs from a healthy donor, v3 chemistry](https://www.10xgenomics.com/datasets/10-k-pbm-cs-from-a-healthy-donor-v-3-chemistry-3-standard-3-0-0), 10x Genomics, Cell Ranger 3.0.0. Source data: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The archive checksum follows [Scanpy’s PBMC tutorial](https://scanpy.scverse.org/en/stable/tutorials/experimental/pearson_residuals.html). Counts remain sparse; preparation and filtering are shown above. The web page reads prepared inputs. Running the notebook downloads and prepares the same dataset.
:::
